# SwitchBoard — Google Colab GPU Full Curriculum Training & Evaluation
**Hindi–English Code-Switching Speech Recognition and Switch-Boundary Error Analysis**

This notebook runs full-scale GPU training (`fp16=True`, `r=16`, `lora_alpha=32`) and full 517-utterance held-out test evaluation on a Google Colab GPU (T4 / L4 / A100) using the unified SwitchBoard codebase:
1. **Primary Baseline:** `openai/whisper-small`
2. **Comparison Baselines:** `vasista22/whisper-hindi-small` (IndicWhisper) & `Harveenchadha/vakyansh-wav2vec2-hindi-him-4200` (wav2vec 2.0 CTC)
3. **Experiment A (LoRA Alone):** `configs/train_colab_full_lora_only.yaml`
4. **Experiment B (LoRA + Auxiliary LID Head):** `configs/train_colab_full_lora_lid.yaml`

In [ ]:
# 1. Verify Colab GPU & Install Dependencies
!nvidia-smi
!pip install -q torch torchaudio transformers peft accelerate streamlit soundfile librosa scipy numpy pandas PyYAML pytest

In [ ]:
# 2. Mount Google Drive or Upload SwitchBoard Repository + Hindi-English_test.tar.gz
import os, torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

In [ ]:
# 3. Validate the 3,136-utterance MUCS 2021 Speaker-Disjoint Manifest
!python -m switchboard.cli validate-manifest --manifest data/full_corpus/unified_manifest.jsonl --check-audio --forbid-demo

In [ ]:
# 4. Run Full 3-Stage Curriculum Fine-Tuning: LoRA Alone vs. LoRA + Auxiliary LID Head
!python -m switchboard.cli train --config configs/train_colab_full_lora_only.yaml --output-dir checkpoints/whisper_small_lora_only
!python -m switchboard.cli train --config configs/train_colab_full_lora_lid.yaml --output-dir checkpoints/whisper_small_lora_plus_lid

In [ ]:
# 5. Evaluate All Real Models on All 517 Held-Out MUCS 2021 Test Utterances
!python -m switchboard.cli evaluate-real-benchmark --manifest data/full_corpus/unified_manifest.jsonl --max-utterances 517 --output-dir outputs/full_benchmark_real --run-tier full_research

In [ ]:
# 6. Display Final Real Model Comparison Table
from pathlib import Path
print(Path('outputs/full_benchmark_real/model_comparison.md').read_text(encoding='utf-8'))